# Example 8.5.1 / 1: Request a Primal Ray

Companion notebook for **8.5.1 Primal and Dual Rays** in the AMPL lab Documentation, section "Request a Primal Ray". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Request a Primal Ray

In this version of the steel model the market bounds were left out and `Time` was typed with
`>=` instead of `<=`, so nothing limits production. `option presolve 0;` sends the original rows
to HiGHS, so presolve cannot simplify the model before the driver returns the ray. In `data.py`
and `run.py`, `ampl` is an `AMPL` object that has already read `model.mod`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                                 # products
param rate {PROD} > 0;                    # tons produced per hour
param avail >= 0;                         # hours available
param profit {PROD};                      # profit per ton
var Make {p in PROD} >= 0;                # tons produced: market bound forgotten
maximize Total_Profit: sum {p in PROD} profit[p] * Make[p];  # total profit
# typo: >= instead of <=
subject to Time: sum {p in PROD} (1 / rate[p]) * Make[p] >= avail;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["rate"] = {"bands": 200, "coils": 140, "plate": 160}
ampl.param["profit"] = {"bands": 25, "coils": 30, "plate": 29}
ampl.param["avail"] = 40

In [5]:
# run.py
ampl.option["presolve"] = 0
ampl.solve(solver="highs", highs_options="alg:rays=1", verbose=False)
print(ampl.solve_result)
print(ampl.get_data("Make.unbdd").to_pandas())

unbounded
       Make.unbdd
bands         200
coils           0
plate           0


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROD := bands coils plate;
param rate := bands 200  coils 140  plate 160;
param profit := bands 25  coils 30  plate 29;
param avail := 40;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option presolve 0;
option solver highs;
option highs_options 'alg:rays=1';
option solver_msg 0;
solve;
display solve_result, Make.unbdd;

HiGHS 1.14.0:   alg:rays = 1

suffix unbdd OUT;
solve_result = unbounded

Make.unbdd [*] :=
bands  200
coils    0
plate    0
;

